In [17]:
from openai import OpenAI
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from typing import Literal

In [5]:
load_dotenv()

True

In [35]:
url = "https://justjoin.it/job-offer/hsbc-service-delivery-senior-software-engineer-epricing--krakow-java"

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:151.0) Gecko/20100101 Firefox/151.0"})
soup = BeautifulSoup(response.text, 'lxml')
offer_text = soup.find("body").text

In [37]:
class OfferData(BaseModel):
    title: str = Field(description="Tytuł oferty pracy")
    company: str = Field(description="Nazwa firmy (pracodawcy)")
    category: str = Field(description="Kategoria oferty, np. Frontend, Java, C++")
    salary_range: list[int] | None = Field(
        description="Minimalne i maksymalne wynagrodzenie na umowie o pracę lub B2B. Jeśli dostępne są obie opcje, wybierz dowolną. Jeśli nie są dostępne widełki, pozostaw None"
    )
    work_model: Literal["remote", "hybrid", "office"] | None = Field(description="Praca zdalna, hybrydowa lub z biura. Jeśli nie ma tej informacji pozostaw None")
    seniority: Literal["junior", "mid", "senior"] | None = Field(description="Seniority")

In [38]:
client = OpenAI()

In [39]:
SYSTEM_PROMPT = """Dostaniesz opis oferty pracy, który jest całym tekstem z HTML strony z ofertą.
Wyciągnij z tego tekstu wszystkie niezbędne informacje na temat oferty, takie jak:
- title
- company
- category
- salary_range
- work_model
- seniority

Na stronie mogą znajdować się informacje pochodzące z innych ofert więc zwróć uwagę żeby wyciągnąć tylko to, co dotyczy
właściwej oferty. Zwróć wynik jako JSON.
"""

In [40]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": offer_text}
]

In [41]:
response = client.responses.parse(
    input=messages,
    model="gpt-4o",
    text_format=OfferData
)

In [42]:
response.output_parsed.model_dump()

{'title': 'Senior Software Engineer (ePricing)',
 'company': 'HSBC Service Delivery',
 'category': 'Java',
 'salary_range': [23300, 32000],
 'work_model': 'hybrid',
 'seniority': 'senior'}